# The Probability Ranking Principle

This notebook is the *narrative* pillar: it imports the reference implementation in `probability_ranking_principle.py` — which **owns every number** — and walks the topic section by section. The headline check brute-forces the theorem over *all* permutations of a small corpus, so optimality is verified, not asserted.

```bash
uv run --with numpy --with jupyter \
    jupyter execute notebooks/probability-ranking-principle/01_probability_ranking_principle.ipynb
```

In [ ]:
import sys, pathlib
_cands = [pathlib.Path.cwd(),
          pathlib.Path.cwd() / "notebooks" / "probability-ranking-principle",
          pathlib.Path("notebooks/probability-ranking-principle")]
for _d in _cands:
    if (_d / "probability_ranking_principle.py").exists():
        sys.path.insert(0, str(_d)); break
import probability_ranking_principle as prp
print("loaded reference implementation; query =", prp._QUERY)

## 1. The decision-theoretic setup

Each document $d$ has a probability of relevance $p_d = P(R=1 \mid d, q)$. By linearity of expectation, the expected number of relevant documents in the top $k$ of an ordering is just the prefix sum of these probabilities — **no independence assumption** is needed.

In [ ]:
p = prp._P
for d, pr in zip(prp._DOCS, p):
    print(f'  {d:<16} P(R)={pr:.2f}')
print('expected relevant in top-3 of the PRP order =',
      round(prp.expected_relevant_at_k(prp.prp_order(p), p, 3), 2))

## 2. The theorem, brute-forced

Ranking by decreasing $P(R)$ maximizes expected relevant@k — and minimizes expected cost — **at every cutoff simultaneously**. We check it exhaustively against all $5! = 120$ permutations, then against several linear cost settings.

In [ ]:
prp.test_prp_maximizes_over_all_permutations()
prp.test_prp_minimizes_cost_over_all_permutations()

## 3. The exchange argument

The proof turns on a one-line lemma: swapping any *out-of-order adjacent pair* never lowers expected relevant@k at any cutoff, and strictly raises it at the cutoff between the pair. Iterating the swap is bubble sort — inversions fall monotonically to zero, ending at the PRP order.

In [ ]:
prp.test_adjacent_swap_lemma()
prp.test_bubble_sort_terminates_at_prp()

## 4. Precision, recall, and the handoff to BM25

Under the 1/0 cost model the principle maximizes expected precision@k and recall@k for every $k$. And because ranking is invariant under strictly increasing transforms, ranking by $P(R)$ is ranking by the **odds** $P/(1-P)$ and by the **log-odds** $\log P/(1-P)$ — which is exactly where the Binary Independence Model and BM25 begin.

In [ ]:
prp.test_precision_recall_special_case()
prp.test_monotone_transform_invariance()

## 5. A finance-flavored worked example

A length-biased retriever floats the long transcript and the wordier items to the top and buries the terse on-point filing below the cutoff; the PRP order, ranking by actual relevance, beats it on expected relevant@3.

In [ ]:
prp.worked_example()
prp.test_prp_beats_plausible_alternative()

## 6. Where the theorem breaks: the additivity assumption

The optimality proof assumes the cost of a ranking is **additive** across documents. When relevance is interdependent — two near-duplicate documents that are each relevant alone but redundant together — that assumption is false, and the PRP is *no longer optimal*. This is the diversity / MMR regime, demonstrated here with a redundancy-discounted utility.

In [ ]:
prp.test_independence_break_diversity()

---

**Three pillars, one set of numbers.** The proof on the page, the interactive exchange-argument laboratory, and this notebook all read from `probability_ranking_principle.py`. The theorem is not asserted here — it is checked against every permutation.